# Post-Training Step 1: Supervised Fine-Tuning (SFT)

**Paper §3.1** — SFT teaches the model reasoning, agentic tool use, instruction
following, safety, and multilingual abilities using ≈ 18 M samples.
Only assistant / response tokens contribute to the loss.

### Key design choices
| Feature | Detail |
|---|---|
| Loss mask | user/system tokens = 0, assistant tokens = 1 |
| Reasoning on/off | 10 % of samples have `<think>` trace stripped (§3.1.5) |
| Budget control | 3 % of samples have a truncated reasoning trace (§3.1.5) |
| Schedule | 13 000 steps, LR 5e-5, 800-step warmup |
| MoE | load-balance auxiliary loss active |

### Notebook contents
1. Environment setup
2. Imports & hyperparameters
3. Tokenizer
4. Dataset (GSM8K as stand-in)
5. Model (load pretrain checkpoint)
6. Optimizer
7. Training loop
8. Evaluation


## 0. Environment Setup

Detects Colab / Kaggle, installs packages, and adds the repo to `sys.path`.

In [ ]:
import os, sys

IN_COLAB  = False
IN_KAGGLE = os.path.exists("/kaggle/working") and os.path.exists("/kaggle/input")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}")


In [ ]:
if IN_COLAB or IN_KAGGLE:
    # Set ACCELERATOR to "cuda12" (GPU) or "tpu" (Colab TPU only).
    ACCELERATOR = "cuda12"
    import subprocess
    subprocess.run(
        ["pip", "install", "-q",
         f"jax[{ACCELERATOR}]", "flax", "optax", "orbax-checkpoint",
         "datasets", "transformers", "matplotlib"],
        check=True,
    )


In [ ]:
import pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-jax-nemotron.git"


def _detect_repo_root() -> pathlib.Path:
    """Detect local repo root by searching upward for key project files."""
    cwd = pathlib.Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "training_shared.py").exists() and (candidate / "nemotron.py").exists():
            return candidate
    return cwd


if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
elif IN_KAGGLE:
    REPO_DIR = pathlib.Path("/kaggle/working/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    REPO_DIR = _detect_repo_root()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("REPO_DIR:", REPO_DIR)


## 1. Imports & Hyperparameters

In [ ]:
import pathlib

import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx
from transformers import AutoTokenizer

from moe import SparseMoE
from nemotron import NemotronConfig, NemotronNanoBlock
from training_shared import (
    PRETRAIN_CKPT_DIR,
    SFT_SEQ_LEN, SFT_BATCH, SFT_STEPS,
    SFT_LR, SFT_MIN_LR, SFT_WARMUP_STEPS,
    SFT_WD, SFT_B1, SFT_B2,
    SFT_CKPT_DIR, SFT_CKPT_EVERY,
    build_model, collect_moe_layers, make_decayed_lr_optimizer,
    load_sft_data, make_sft_batches, sft_step, sft_loss,
    update_moe_biases, make_checkpoint_manager, save_checkpoint,
    try_load_from_dir,
)

print("JAX devices:", jax.devices())


In [ ]:
SEQ_LEN    = SFT_SEQ_LEN   # 256
BATCH_SIZE = SFT_BATCH     # 2
TRAIN_STEPS = SFT_STEPS    # 300
CKPT_DIR    = SFT_CKPT_DIR

print(f"SFT: SEQ_LEN={SEQ_LEN} | BATCH={BATCH_SIZE} | STEPS={TRAIN_STEPS}")
print(f"LR schedule: peak={SFT_LR}, min={SFT_MIN_LR}, warmup={SFT_WARMUP_STEPS}")


## 2. Tokenizer

In [ ]:
print("Loading Nemotron tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size : {tokenizer.vocab_size}")


## 3. Dataset

GSM8K is used as a stand-in for the paper's full SFT dataset of ≈ 18 M samples.
Each sample is formatted as a `<think>` reasoning trace followed by the final answer.
10 % of samples have the `<think>` block stripped to enable *reasoning off* mode.

In [ ]:
train_samples = load_sft_data("train")
val_samples   = load_sft_data("test")

print(f"Train samples : {len(train_samples)}")
print(f"Val   samples : {len(val_samples)}")


## 4. Model — Load Pretrain Checkpoint

In [ ]:
print("Building model ...")
model = build_model(seed=0)
moe_layers = collect_moe_layers(model)

loaded = try_load_from_dir(PRETRAIN_CKPT_DIR, model, model.config)
if loaded:
    print("Resumed from pretrain checkpoint.")
else:
    print("No pretrain checkpoint found — fine-tuning from random init.")


## 5. Optimizer

In [ ]:
tx = make_decayed_lr_optimizer(
    peak_lr=SFT_LR,
    min_lr=SFT_MIN_LR,
    warmup_steps=SFT_WARMUP_STEPS,
    stable_steps=max(1, TRAIN_STEPS - SFT_WARMUP_STEPS - 1),
    decay_steps=1,
    weight_decay=SFT_WD,
    b1=SFT_B1,
    b2=SFT_B2,
)
optimizer  = nnx.Optimizer(model, tx, wrt=nnx.Param)
ckpt_mgr   = make_checkpoint_manager(CKPT_DIR)
print("Optimizer and checkpoint manager created.")


## 6. Training Loop

> Only assistant tokens (where `mask == 1`) contribute to the loss.

In [ ]:
print(f"\n=== Post-Training Step 1: SFT ===")
print(f"Training for {TRAIN_STEPS} steps ...\n")

loss_history: list[float] = []
step = 0

# Cache a small fixed validation window once to avoid repeated tokenisation.
val_eval_batches = []
for vinputs, vlabels, vmask in make_sft_batches(val_samples, tokenizer, BATCH_SIZE, SEQ_LEN):
    val_eval_batches.append((vinputs, vlabels, vmask))
    if len(val_eval_batches) >= 10:
        break

while step < TRAIN_STEPS:
    for inputs, labels, mask in make_sft_batches(train_samples, tokenizer, BATCH_SIZE, SEQ_LEN):
        if step >= TRAIN_STEPS:
            break
        loss = sft_step(model, optimizer, inputs, labels, mask)
        update_moe_biases(moe_layers)
        step += 1
        loss_history.append(float(loss))

        if step % 50 == 0:
            val_loss = 0.0
            for vinputs, vlabels, vmask in val_eval_batches:
                val_loss += float(sft_loss(model, vinputs, vlabels, vmask, moe_layers))
            val_loss /= max(len(val_eval_batches), 1)
            print(f"  SFT step {step:4d} | train_loss={float(loss):.4f} | val_loss={val_loss:.4f}")

        if step % SFT_CKPT_EVERY == 0:
            save_checkpoint(ckpt_mgr, model, step)

save_checkpoint(ckpt_mgr, model, step)
print("\nSFT complete.")


## 7. Loss Curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history)
    plt.xlabel("Step"); plt.ylabel("SFT Loss"); plt.title("SFT Training Loss")
    plt.grid(True); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")
